In [ ]:
%pip -q install transformers==4.44.2 datasets==3.0.1 accelerate==0.34.2 evaluate==0.4.2 scikit-learn==1.5.2

In [ ]:
import os, random, re
import numpy as np
import pandas as pd
import xml.etree.ElementTree as ET
from typing import List, Dict

import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

from sklearn.metrics import accuracy_score, f1_score, precision_recall_fscore_support, classification_report

from transformers import (
    AutoTokenizer, AutoConfig, AutoModelForSequenceClassification,
    DataCollatorWithPadding, Trainer, TrainingArguments, EarlyStoppingCallback
)

# ----------------- Repro & Device -----------------
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)

# ----------------- Labels -----------------
TARGET_LABELS: List[str] = [
    "anger", "disgust", "fear", "joy", "neutral", "sadness", "surprise"
]
label2id: Dict[str, int] = {l: i for i, l in enumerate(TARGET_LABELS)}
id2label: Dict[int, str] = {i: l for l, i in label2id.items()}

# ----------------- Loaders for each source -----------------
def clean_text(s):
    return str(s).replace("\n", " ").replace("\r", " ").strip()

def load_emotionalturk():
    df = pd.read_csv("/kaggle/input/emotionalturk/EmotionalTurk.csv")
    if not {"text", "label"}.issubset(df.columns):
        raise ValueError(f"EmotionalTurk must contain 'text' and 'label'. Got: {df.columns.tolist()}")
    df["label"] = df["label"].str.lower().str.strip()
    df["text"]  = df["text"].map(clean_text)
    df = df[df["label"].isin(TARGET_LABELS)].reset_index(drop=True)
    return df

def load_turkish_tweets():
    df = pd.read_excel("/kaggle/input/turkish-tweet-dataset/TurkishTweets.xlsx", sheet_name="Sayfa1")
    if not {"Tweet", "Etiket"}.issubset(df.columns):
        raise ValueError(f"TurkishTweets must contain 'Tweet' and 'Etiket'. Got: {df.columns.tolist()}")
    
    label_map = {
        'kızgın': 'anger',
        'korku': 'fear',
        'mutlu': 'joy',
        'surpriz': 'surprise',
        'üzgün': 'sadness'
    }
    
    df["label"] = df["Etiket"].str.lower().str.strip().map(label_map)
    df["text"]  = df["Tweet"].map(clean_text)
    df = df[["text", "label"]]
    df = df[df["label"].isin(TARGET_LABELS)].reset_index(drop=True)
    return df

def load_emotion_dataset():
    train_df = pd.read_csv("/kaggle/input/turkish-emotion-dataset/Emotion_dataset_train.csv")
    test_df  = pd.read_csv("/kaggle/input/turkish-emotion-dataset/Emotion_dataset_test.csv")
    
    df = pd.concat([train_df, test_df], ignore_index=True)
    
    if "label" not in df.columns:
        raise ValueError(f"Emotion dataset must contain 'label'. Got: {df.columns.tolist()}")
    
    # Find text column
    text_col = None
    for col in df.columns:
        if col.lower() in ['sentence', 'text', 'tweet']:
            text_col = col
            break
    if text_col is None:
        raise ValueError(f"No text column found in: {df.columns.tolist()}")
    
    label_map = {
        'kızgın': 'anger',
        'korku': 'fear',
        'mutlu': 'joy',
        'surpriz': 'surprise',
        'üzgün': 'sadness'
    }
    
    df["label"] = df["label"].map(label_map)
    df["text"]  = df[text_col].map(clean_text)
    df = df[["text", "label"]]
    df = df[df["label"].isin(TARGET_LABELS)].reset_index(drop=True)
    return df

def load_tremo():
    tree = ET.parse("/kaggle/input/tremo/TREMODATA.xml")
    root = tree.getroot()
    
    data = []
    for doc in root.findall('Doc'):
        text = doc.find('Entry').text
        emotion = doc.find('ValidatedEmotion').text
        data.append({'text': text, 'emotion': emotion})
    
    df = pd.DataFrame(data)
    
    label_map = {
        'Happy': 'joy',
        'Sadness': 'sadness',
        'Anger': 'anger',
        'Fear': 'fear',
        'Disgust': 'disgust',
        'Surprise': 'surprise'
    }
    
    df["label"] = df["emotion"].map(label_map)
    df["text"]  = df["text"].map(clean_text)
    df = df[["text", "label"]]
    df = df[df["label"].isin(TARGET_LABELS)].reset_index(drop=True)
    return df

def load_duygular():
    df = pd.read_csv("/kaggle/input/trke-cmleler-ve-duygular-duygu-analizi/duygular.csv")
    if not {"text", "emotion"}.issubset(df.columns):
        raise ValueError(f"Duygular must contain 'text' and 'emotion'. Got: {df.columns.tolist()}")
    
    label_map = {
        'Üzüntü': 'sadness',
        'İğrenme': 'disgust',
        'Nötr': 'neutral',
        'Korku': 'fear',
        'Mutluluk': 'joy',
        'Şaşkınlık': 'surprise',
        'Öfke': 'anger'
    }
    
    df["label"] = df["emotion"].map(label_map)
    df["text"]  = df["text"].map(clean_text)
    df = df[["text", "label"]]
    df = df[df["label"].isin(TARGET_LABELS)].reset_index(drop=True)
    return df

# ----------------- Build Combined Splits -----------------
emotionalturk_df = load_emotionalturk()
tweets_df = load_turkish_tweets()
emotion_df = load_emotion_dataset()
tremo_df = load_tremo()
duygular_df = load_duygular()

# Combine all
all_df = pd.concat([emotionalturk_df, tweets_df, emotion_df, tremo_df, duygular_df], ignore_index=True)

# Split: 70% train, 15% dev, 15% test
from sklearn.model_selection import train_test_split
train_df, temp_df = train_test_split(all_df, test_size=0.3, stratify=all_df["label"], random_state=SEED)
dev_df, test_df = train_test_split(temp_df, test_size=0.5, stratify=temp_df["label"], random_state=SEED)

print("Train size:", len(train_df), " Dev size:", len(dev_df), " Test size:", len(test_df))
print("\nTrain label distribution:")
print(train_df["label"].value_counts().sort_index())

# ----------------- Dataset -----------------
class TextDataset(Dataset):
    def __init__(self, df, tokenizer, max_length=128):
        self.texts = df["text"].tolist()
        self.labels = [label2id[l] for l in df["label"].tolist()]
        self.tok = tokenizer
        self.max_length = max_length

    def __len__(self): return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tok(
            self.texts[idx],
            max_length=self.max_length,
            truncation=True,
            padding=False
        )
        enc["labels"] = self.labels[idx]
        return enc

# ----------------- Tokenizer / Model -----------------
MODEL_NAME = "roberta-base"

import os
os.environ["TRANSFORMERS_OFFLINE"] = "0"

print(f"Downloading {MODEL_NAME}...")
from huggingface_hub import snapshot_download
try:
    snapshot_download(repo_id=MODEL_NAME, allow_patterns=["*.json", "*.txt", "*.bin", "*.safetensors", "*.model"])
    print("Files downloaded")
except:
    pass  

# Now load with local_files_only to avoid the bug
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True, local_files_only=True)
print("Tokenizer loaded")

config = AutoConfig.from_pretrained(
    MODEL_NAME,
    num_labels=len(TARGET_LABELS),
    id2label=id2label,
    label2id=label2id,
    problem_type="single_label_classification",
    local_files_only=True
)

model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, config=config, local_files_only=True).to(DEVICE)
print("Model loaded")
collator = DataCollatorWithPadding(tokenizer=tokenizer)

train_ds = TextDataset(train_df, tokenizer, max_length=128)
dev_ds   = TextDataset(dev_df,   tokenizer, max_length=128)
test_ds  = TextDataset(test_df,  tokenizer, max_length=128)

# ----------------- Class Weights & Focal Loss -----------------
train_labels_np = np.array([label2id[lbl] for lbl in train_df["label"].values], dtype=np.int64)
class_counts = np.bincount(train_labels_np, minlength=len(TARGET_LABELS))
total = class_counts.sum()
class_weights = total / (len(TARGET_LABELS) * class_counts.clip(min=1))
class_weights = class_weights / class_weights.mean()
class_weights_t = torch.tensor(class_weights, dtype=torch.float32, device=DEVICE)
print("\nClass counts:", class_counts.tolist())
print("Class weights (normalized):", class_weights.tolist())

class FocalLoss(nn.Module):
    def __init__(self, weight=None, gamma=2.0, reduction="mean"):
        super().__init__()
        self.weight = weight
        self.gamma = gamma
        self.reduction = reduction
        self.ce = nn.CrossEntropyLoss(weight=weight, reduction="none")

    def forward(self, logits, targets):
        ce_loss = self.ce(logits, targets)
        with torch.no_grad():
            probs = torch.softmax(logits, dim=-1)
            pt = probs[torch.arange(logits.size(0), device=logits.device), targets].clamp_min(1e-8)
        focal = (1 - pt) ** self.gamma * ce_loss
        if self.reduction == "mean":
            return focal.mean()
        elif self.reduction == "sum":
            return focal.sum()
        else:
            return focal

# ----------------- WeightedRandomSampler -----------------
per_class_inv = 1.0 / class_counts.clip(min=1)
sample_weights = per_class_inv[train_labels_np]
sample_weights = torch.tensor(sample_weights, dtype=torch.double)
sampler = WeightedRandomSampler(
    weights=sample_weights,
    num_samples=len(sample_weights),
    replacement=True
)

# ----------------- Custom Trainer -----------------
from transformers import Trainer

class FocalLossTrainer(Trainer):
    def __init__(self, *args, focal_gamma=2.0, class_weights=None, **kwargs):
        super().__init__(*args, **kwargs)
        self.focal_loss = FocalLoss(weight=class_weights, gamma=focal_gamma)

    def get_train_dataloader(self):
        return DataLoader(
            self.train_dataset,
            batch_size=self.args.train_batch_size,
            sampler=sampler,
            collate_fn=self.data_collator,
            pin_memory=True
        )

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits
        loss = self.focal_loss(logits, labels)
        return (loss, outputs) if return_outputs else loss

# ----------------- Metrics -----------------
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = logits.argmax(axis=-1)
    acc = accuracy_score(labels, preds)
    p, r, f1, _ = precision_recall_fscore_support(labels, preds, average="macro", zero_division=0)
    f1w = precision_recall_fscore_support(labels, preds, average="weighted", zero_division=0)[2]
    return {"accuracy": acc, "precision_macro": p, "recall_macro": r, "f1_macro": f1, "f1_weighted": f1w}

# ----------------- Training Args -----------------
OUTPUT_DIR = "/kaggle/working/turkish-emotion-roberta-final"
args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="steps",
    logging_steps=200,
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    greater_is_better=True,
    num_train_epochs=5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    gradient_accumulation_steps=1,
    learning_rate=3e-5,
    weight_decay=0.01,
    warmup_ratio=0.1,
    fp16=torch.cuda.is_available(),
    report_to="none",
    save_total_limit=2,
    seed=SEED
)

trainer = FocalLossTrainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=dev_ds,
    tokenizer=tokenizer,
    data_collator=collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
    focal_gamma=2.0,
    class_weights=class_weights_t
)

train_result = trainer.train()
print("Best checkpoint:", trainer.state.best_model_checkpoint)

# ----------------- Final eval on test -----------------
metrics = trainer.evaluate(test_ds)
print("Test metrics:", metrics)

# Save final artifacts
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

# ----------------- Per-label report -----------------
def predict_all(ds, batch_size=64):
    preds, golds = [], []
    model.eval()
    with torch.no_grad():
        for i in range(0, len(ds), batch_size):
            batch = [ds[j] for j in range(i, min(i+batch_size, len(ds)))]
            batch = collator(batch)
            batch = {k: torch.tensor(v).to(DEVICE) for k, v in batch.items()}
            logits = model(**{k: v for k, v in batch.items() if k != "labels"}).logits
            preds.extend(logits.argmax(-1).detach().cpu().tolist())
            golds.extend(batch["labels"].detach().cpu().tolist())
    return golds, preds

golds, preds = predict_all(test_ds)
print("\n=== Classification Report (Test) ===")
print(classification_report(golds, preds, target_names=TARGET_LABELS, digits=4))

2025-10-12 07:58:13.131516: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1760255893.338458     116 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1760255893.395745     116 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


Device: cuda
Train size: 33632  Dev size: 7207  Test size: 7207

Train label distribution:
label
anger       5228
disgust     4306
fear        5410
joy         6056
neutral     2014
sadness     6491
surprise    4127
Name: count, dtype: int64
📦 Downloading roberta-base...


Fetching 8 files:   0%|          | 0/8 [00:00<?, ?it/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

...

Epoch,Training Loss,Validation Loss,Accuracy,Precision Macro,Recall Macro,F1 Macro,F1 Weighted
1,0.305300,0.278548,0.799084,0.796824,0.822001,0.798854,0.801933
2,0.184900,0.212573,0.849868,0.846056,0.865739,0.853001,0.849754
3,0.120300,0.195496,0.865409,0.863435,0.876705,0.867948,0.865151
4,0.080600,0.199955,0.868045,0.860801,0.883707,0.868636,0.867229
5,0.057700,0.181194,0.889413,0.887101,0.897781,0.891980,0.889169


Best checkpoint: /kaggle/working/turkish-emotion-roberta-final/checkpoint-10510


Test metrics: {'eval_loss': 0.19022656977176666, 'eval_accuracy': 0.892743166366033, 'eval_precision_macro': 0.8909474527752168, 'eval_recall_macro': 0.8967202376025235, 'eval_f1_macro': 0.8936638171835803, 'eval_f1_weighted': 0.8925525803420645, 'eval_runtime': 18.1989, 'eval_samples_per_second': 396.013, 'eval_steps_per_second': 12.418, 'epoch': 5.0}


/tmp/ipykernel_116/157880535.py:358: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  batch = {k: torch.tensor(v).to(DEVICE) for k, v in batch.items()}



=== Classification Report (Test) ===
              precision    recall  f1-score   support

       anger     0.8837    0.8608    0.8721      1121
     disgust     0.9200    0.9469    0.9333       923
        fear     0.8959    0.9137    0.9047      1159
         joy     0.8979    0.8882    0.8930      1297
     neutral     0.8568    0.9005    0.8781       432
     sadness     0.8768    0.8548    0.8657      1391
    surprise     0.9067    0.9129    0.9098       884

    accuracy                         0.8929      7207
   macro avg     0.8911    0.8968    0.8938      7207
weighted avg     0.8928    0.8929    0.8927      7207



In [2]:
!zip -r /kaggle/working/turkish-emotion-roberta-final.zip /kaggle/working/turkish-emotion-roberta-final

  adding: kaggle/working/turkish-emotion-roberta-final/ (stored 0%)
  adding: kaggle/working/turkish-emotion-roberta-final/checkpoint-8408/ (stored 0%)
  adding: kaggle/working/turkish-emotion-roberta-final/checkpoint-8408/trainer_state.json (deflated 73%)
  adding: kaggle/working/turkish-emotion-roberta-final/checkpoint-8408/config.json (deflated 54%)
  adding: kaggle/working/turkish-emotion-roberta-final/checkpoint-8408/merges.txt (deflated 53%)
  adding: kaggle/working/turkish-emotion-roberta-final/checkpoint-8408/rng_state.pth (deflated 25%)
  adding: kaggle/working/turkish-emotion-roberta-final/checkpoint-8408/training_args.bin (deflated 51%)
  adding: kaggle/working/turkish-emotion-roberta-final/checkpoint-8408/optimizer.pt

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


 (deflated 34%)
  adding: kaggle/working/turkish-emotion-roberta-final/checkpoint-8408/vocab.json (deflated 59%)
  adding: kaggle/working/turkish-emotion-roberta-final/checkpoint-8408/tokenizer.json (deflated 72%)
  adding: kaggle/working/turkish-emotion-roberta-final/checkpoint-8408/tokenizer_config.json (deflated 76%)
  adding: kaggle/working/turkish-emotion-roberta-final/checkpoint-8408/scheduler.pt (deflated 56%)
  adding: kaggle/working/turkish-emotion-roberta-final/checkpoint-8408/model.safetensors (deflated 15%)
  adding: kaggle/working/turkish-emotion-roberta-final/checkpoint-8408/special_tokens_map.json (deflated 52%)
  adding: kaggle/working/turkish-emotion-roberta-final/config.json (deflated 54%)
  adding: kaggle/working/turkish-emotion-roberta-final/merges.txt (deflated 53%)
  adding: kaggle/working/turkish-emotion-roberta-final/training_args.bin (deflated 51%)
  adding: kaggle/working/turkish-emotion-roberta-final/vocab.json (deflated 59%)
  adding: kaggle/working/turkish-